# Real Molecular Mechanics Core

This notebook exercises the new system-level molecular mechanics surface: typed systems, force-field assignment, combined nonbonded interactions, constraints, trajectory I/O, restart, and benchmark rows. The point is to verify that the pieces work together as a real workflow.

In [ ]:
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.benchmarks import mm_force_terms
from mlx_atomistic.diagnostics import summarize_md_result
from mlx_atomistic.examples import mixed_lj_fluid_example, water_like_constrained_example
from mlx_atomistic.io import (
    load_npz_trajectory,
    read_xyz,
    restart_state_from_trajectory,
    save_npz_trajectory,
    write_xyz,
)
from mlx_atomistic.md import SimulationConfig, simulate_nve
from mlx_atomistic.runtime import get_runtime_info

get_runtime_info()

## 1. Typed system and force-field assignment

The water-like example is intentionally small. It still has the ingredients we need for real MM infrastructure: atom names, atom types, masses, charges, topology, force-field parameters, nonbonded terms, and constraints.

In [ ]:
system, force_field, constraints = water_like_constrained_example()
force_terms = force_field.build_force_terms(system)

{
    "symbols": system.symbols,
    "atom_types": system.atom_types,
    "masses": np.array(system.masses).round(4).tolist(),
    "charges": np.array(system.charges).round(4).tolist(),
    "force_terms": [term.name for term in force_terms],
}

## 2. Constrained NVE run

The distance constraints keep the O-H distances fixed while the rest of the dynamics and diagnostics continue to work. Constraint error is recorded densely at every integration step.

In [ ]:
result = simulate_nve(
    system.positions,
    system.velocities,
    masses=system.masses,
    force_terms=force_terms,
    config=SimulationConfig(dt=0.001, steps=20, sample_interval=2),
    constraints=constraints,
)
summarize_md_result(result)

In [ ]:
steps = np.arange(result.total_energy.shape[0])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(steps, np.array(result.total_energy), label="total")
for name, series in result.potential_energy_by_term.items():
    axes[0].plot(steps, np.array(series), label=name)
axes[0].set_xlabel("step")
axes[0].set_ylabel("energy")
axes[0].set_title("Energy decomposition")
axes[0].legend(fontsize=8)

axes[1].plot(steps, np.array(result.constraint_max_error))
axes[1].set_xlabel("step")
axes[1].set_ylabel("max distance error")
axes[1].set_title("Constraint error")
fig.tight_layout()

## 3. Structure and trajectory I/O

XYZ gives us a simple structure interchange path. The native NPZ trajectory keeps sampled frames, dense scalar diagnostics, energy decomposition, symbols, optional cell data, and JSON metadata. A restart helper recomputes forces from a saved frame.

In [ ]:
with TemporaryDirectory() as tmp:
    xyz_path = f"{tmp}/water.xyz"
    traj_path = f"{tmp}/water-traj.npz"
    write_xyz(xyz_path, system.symbols, system.positions, comment="water-like")
    xyz_symbols, xyz_positions, xyz_comment = read_xyz(xyz_path)
    save_npz_trajectory(traj_path, result, symbols=system.symbols, metadata={"case": "water"})
    record = load_npz_trajectory(traj_path)
    restart = restart_state_from_trajectory(record, system.masses, force_terms)

{
    "xyz_symbols": xyz_symbols,
    "xyz_comment": xyz_comment,
    "xyz_shape": xyz_positions.shape,
    "frames": record.sampled_positions.shape[0],
    "metadata": record.metadata,
    "restart_step": restart.step,
}

## 4. Mixed typed LJ fluid

The mixed fluid uses atom types with different `σ` and `ε`. The combined nonbonded path applies Lorentz-Berthelot mixing for every pair, which is the first realistic pair path we should benchmark before thinking about custom Metal kernels.

In [ ]:
mixed_system, mixed_force_field = mixed_lj_fluid_example(particles=32)
mixed_terms = mixed_force_field.build_force_terms(mixed_system)
energy, forces = mixed_terms[-1].energy_forces(mixed_system.positions, cell=mixed_system.cell)
{
    "atoms": mixed_system.atom_count,
    "force_terms": [term.name for term in mixed_terms],
    "nonbonded_energy": float(np.array(energy)),
    "force_shape": np.array(forces).shape,
}

## 5. Benchmark rows

The benchmark now includes legacy LJ, direct Coulomb, combined nonbonded, neighbor-list build, bonded autodiff, and constraint projection rows. This is the evidence surface for the future kernel decision.

In [ ]:
bench = [row.to_dict() for row in mm_force_terms.run_benchmark(evaluations=2, particles=32)]
labels = [f"{row['category']}\n{row['term']}" for row in bench]
times = [row["ms_per_eval"] for row in bench]

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(labels, times)
ax.set_ylabel("ms / eval")
ax.set_title("MM force-path and constraint timing")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
bench